In [2]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

## Importing Libraries

In [3]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay)

print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("device", device)

device cuda


In [8]:
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

GPU:  Tesla T4


## Dataset Loading

In [9]:
print(os.listdir("/kaggle/input"))

['datasets']


## Data Loading

In [15]:
import random, shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/sanikamal/horses-or-humans-dataset")
DST = Path("horse_dataset")
random.seed(42)

exts = {".png", ".jpg", ".jpeg"}
pool = {"horse": {}, "not_horse": {}}   # dict by filename removes duplicates

for p in SRC.rglob("*"):
    if p.suffix.lower() in exts:
        folder = p.parent.name.lower()
        if folder == "horses":
            pool["horse"][p.name] = p
        elif folder == "humans":
            pool["not_horse"][p.name] = p

if DST.exists():
    shutil.rmtree(DST)

for cls, files in pool.items():
    files = list(files.values())
    random.shuffle(files)
    n = len(files)
    n_train, n_val = int(0.70 * n), int(0.15 * n)
    splits = {
        "train": files[:n_train],
        "val":   files[n_train:n_train + n_val],
        "test":  files[n_train + n_val:],
    }
    for split, items in splits.items():
        out = DST / split / cls
        out.mkdir(parents=True, exist_ok=True)
        for f in items:
            shutil.copy(f, out / f.name)

for split in ["train", "val", "test"]:
    for cls in ["horse", "not_horse"]:
        print(split, cls, len(list((DST / split / cls).glob("*"))))

train horse 439
train not_horse 458
val horse 94
val not_horse 98
test horse 95
test not_horse 99


## Data pre-processing

In [21]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
    ])

In [22]:
eval_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [23]:
train_dataset = datasets.ImageFolder(
    "horse_dataset/train",
    transform = train_transform
)

val_dataset = datasets.ImageFolder(
    "horse_dataset/val",
    transform = eval_transform
)

test_dataset = datasets.ImageFolder(
    "horse_dataset/test",
    transform = eval_transform
)

In [24]:
print(train_dataset.class_to_idx)

{'horse': 0, 'not_horse': 1}


In [25]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [26]:
images , labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


## Architecture Building

In [31]:
class HorseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            #block 1
            nn.Conv2d(3, 32,kernel_size=3, padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            #block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            #block3 
            nn.Conv2d(64, 128, kernel_size= 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Linear(128,2)

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x

In [32]:
model = HorseCNN().to(device)
print(model)

HorseCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [34]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

## Validation

In [44]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
    
        optimizer.zero_grad()
    
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        predictions = outputs.argmax(dim=1)
    
        correct = correct+(predictions==labels).sum().item()
        total = total+labels.size(0)
    
        total_loss = total_loss+loss.item() * labels.size(0)
    
    epoch_loss = total_loss/ total
    epoch_accuracy = correct / total * 100
    
    return epoch_loss, epoch_accuracy

In [45]:
def validate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            total_loss += loss.item() * labels.size(0)

    epoch_loss = total_loss / total
    epoch_accuracy = correct / total * 100

    return epoch_loss, epoch_accuracy

## Training initialization

In [46]:
epochs = 12

best_val_accuracy = 0

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion
    )

    print(f"Epoch: {epoch+1}/{epochs}")

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_accuracy:

        best_val_accuracy = val_acc

        torch.save(
            model.state_dict(),
            "best_horse_cnn.pth"
        )
        print("Best Model Saved!")
    print("_" * 40)

Epoch: 1/12
Train Loss: 0.5302
Train Accuracy: 74.92%
Val Loss: 0.5060
Val Accuracy: 76.04%
Best Model Saved!
________________________________________
Epoch: 2/12
Train Loss: 0.4093
Train Accuracy: 82.39%
Val Loss: 0.3569
Val Accuracy: 84.90%
Best Model Saved!
________________________________________
Epoch: 3/12
Train Loss: 0.3478
Train Accuracy: 85.06%
Val Loss: 0.4405
Val Accuracy: 80.21%
________________________________________
Epoch: 4/12
Train Loss: 0.3531
Train Accuracy: 86.51%
Val Loss: 0.4670
Val Accuracy: 82.29%
________________________________________
Epoch: 5/12
Train Loss: 0.2924
Train Accuracy: 89.30%
Val Loss: 0.2421
Val Accuracy: 92.19%
Best Model Saved!
________________________________________
Epoch: 6/12
Train Loss: 0.2989
Train Accuracy: 88.18%
Val Loss: 0.5593
Val Accuracy: 81.77%
________________________________________
Epoch: 7/12
Train Loss: 0.3332
Train Accuracy: 85.17%
Val Loss: 0.2942
Val Accuracy: 86.98%
________________________________________
Epoch: 8/12
Tra

 ## Evaluation

In [48]:
model.load_state_dict(
    torch.load(
        "best_horse_cnn.pth",
        map_location = device,
        weights_only=True
    )
)

model.to(device)

test_loss, test_acc = validate(
    model,
    test_loader,
    criterion
)

print("Test Loss: ", test_loss)
print("Test accuracy: ", test_acc)

Test Loss:  0.24715425965097762
Test accuracy:  89.69072164948454


In [49]:
torch.save(
    model.state_dict(),
    "best_horse_cnn.pth"
)

In [50]:
model = HorseCNN()
model.load_state_dict(
    torch.load(
        "best_horse_cnn.pth",
        map_location= device,
        weights_only=True
    )
)

model.to(device)

model.eval()

HorseCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)